# Vertical velocity changes with depth and eddy tilt

This simple cache-only notebook asks whether eddy-days with larger `TiltDis` also show a larger vertical change in core-mean upward velocity. It shows the mean, maximum upward and minimum downward velocity profiles for selected snapshots. The depth-change measure is a **proxy**, not the pointwise vortex-stretching term: the fitted core centre and ellipse move with depth. Model `w` is positive upward and cached `Depth` is positive downward.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

HERE=Path.cwd()
if HERE.name!='vertical_velocity': raise RuntimeError('Launch from the vertical_velocity folder')
sys.path.insert(0,str(HERE.parent))
import seacofs_tilt_tools as tilt


In [ ]:
FRACTION=1.5
MAX_DEPTH_M=1000
UPPER_WINDOW_M=(100,300)
DEEP_WINDOW_M=(500,900)
MIN_LEVELS_PER_WINDOW=2
MIN_CORE_CELLS=8
MIN_COVERAGE=0.7
PROFILE_CASES=None  # Or [(Eddy, Day), ...]
N_PROFILE_DAYS_PER_CLASS=3
SEED=731
CACHE_PATH=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/vertical_velocity_climatology/core_vertical_velocity_upper_1000m.parquet')


In [ ]:
raw=pd.read_parquet(CACHE_PATH)
required={'Eddy','Day','fraction','Depth','n_valid','coverage','w_mean','w_max','w_min'}
missing=required-set(raw.columns)
if missing: raise ValueError(f'Cache missing {sorted(missing)}; rerun notebook 02')
depth=raw.loc[np.isclose(raw.fraction,FRACTION)&raw.Depth.between(0,MAX_DEPTH_M)].copy()
depth=depth.loc[depth.n_valid.ge(MIN_CORE_CELLS)&depth.coverage.ge(MIN_COVERAGE)]
depth=depth.dropna(subset=['w_mean','w_max','w_min'])
if depth.duplicated(['Eddy','Day','Depth']).any(): raise ValueError('Duplicate eddy-day-depth rows')
surface,_=tilt.load_tilt_tables()
identity=surface[['Eddy','Day','Cyc','TiltDis','TiltDir']].drop_duplicates(['Eddy','Day'])
depth=depth.drop(columns=['Cyc','TiltDis','TiltDir'],errors='ignore').merge(identity,on=['Eddy','Day'],how='left',validate='many_to_one')
depth=depth.loc[depth.Cyc.isin(['AE','CE'])].dropna(subset=['TiltDis'])
print('Available cache rows:')
display(depth.groupby('Cyc').agg(eddies=('Eddy','nunique'),depth_rows=('Depth','size'),median_coverage=('coverage','median')))


## One depth-change value per eddy day

Average `w_mean` within 100–300 m and 500–900 m, requiring at least two fitted levels in each window. Divide their difference by the distance between the windows' mean sampled depths. Since depth increases downward, the estimate of the upward-coordinate gradient is `−Δw/ΔDepth`. Positive values mean upward velocity increases toward the surface across these broad windows.


In [ ]:
rows=[]
for (eddy,day),g in depth.groupby(['Eddy','Day'],sort=False):
    upper=g.loc[g.Depth.between(*UPPER_WINDOW_M)]
    deep=g.loc[g.Depth.between(*DEEP_WINDOW_M)]
    if len(upper)<MIN_LEVELS_PER_WINDOW or len(deep)<MIN_LEVELS_PER_WINDOW:continue
    upper_w=upper.w_mean.mean();deep_w=deep.w_mean.mean()
    upper_z=upper.Depth.mean();deep_z=deep.Depth.mean()
    slope_up=-(deep_w-upper_w)/(deep_z-upper_z) # z positive upward; units s^-1
    rows.append(dict(Eddy=eddy,Day=day,Cyc=g.Cyc.iloc[0],TiltDis=g.TiltDis.iloc[0],TiltDir=g.TiltDir.iloc[0],
                     upper_w_mean=upper_w,deep_w_mean=deep_w,
                     upper_depth_m=upper_z,deep_depth_m=deep_z,
                     dwdz_proxy=slope_up,abs_dwdz_proxy=abs(slope_up),
                     upper_levels=len(upper),deep_levels=len(deep)))
snapshot=pd.DataFrame(rows)
if snapshot.empty:raise ValueError('No eddy-days span both depth windows; relax the window or coverage settings')
print('Snapshots spanning both windows:')
display(snapshot.groupby('Cyc').agg(eddies=('Eddy','nunique'),eddy_days=('Day','size'),
                                     median_tilt_km=('TiltDis','median'),
                                     median_abs_gradient=('abs_dwdz_proxy','median')))
display(snapshot.head(10))


## Example eddy-day profiles

Each panel shows cached velocity within the moving fitted core. The pale bands identify the two depth windows used for the broad difference. The maximum and minimum lines show how upward and downward cells can coexist even when their core mean is small.


In [ ]:
if PROFILE_CASES is None:
    cases=[]
    for cyc in ('AE','CE'):
        d=snapshot.loc[snapshot.Cyc.eq(cyc)]
        sample=d.sample(n=min(N_PROFILE_DAYS_PER_CLASS,len(d)),random_state=SEED)
        cases+=list(sample[['Eddy','Day']].itertuples(index=False,name=None))
else:cases=list(PROFILE_CASES)
if not cases:raise ValueError('No profile examples available')
ncols=min(3,len(cases));nrows=int(np.ceil(len(cases)/ncols))
fig,axes=plt.subplots(nrows,ncols,figsize=(4.5*ncols,5*nrows),sharey=True,squeeze=False)
for ax,(eddy,day) in zip(axes.flat,cases):
    g=depth.loc[depth.Eddy.eq(eddy)&depth.Day.eq(day)].sort_values('Depth')
    s=snapshot.loc[snapshot.Eddy.eq(eddy)&snapshot.Day.eq(day)]
    if g.empty or len(s)!=1:raise ValueError(f'Example {(eddy,day)} not in the filtered cache')
    s=s.iloc[0]
    ax.axhspan(*UPPER_WINDOW_M,color='gold',alpha=.12)
    ax.axhspan(*DEEP_WINDOW_M,color='cyan',alpha=.10)
    ax.plot(1e3*g.w_mean,g.Depth,'o-',color='black',label='core mean')
    ax.plot(1e3*g.w_max,g.Depth,color='tab:red',alpha=.7,label='maximum')
    ax.plot(1e3*g.w_min,g.Depth,color='tab:blue',alpha=.7,label='minimum')
    ax.axvline(0,color='.5',lw=.8)
    ax.set(title=f'{s.Cyc} {eddy}, day {day}\nTilt {s.TiltDis:.1f} km; proxy {1e8*s.dwdz_proxy:.2f} ×10⁻⁸ s⁻¹',xlabel='Upward w (mm/s)')
for ax in list(axes.flat)[len(cases):]:ax.set_visible(False)
axes[0,0].set_ylabel('Depth (m)');axes[0,0].invert_yaxis();axes[0,0].legend()
fig.tight_layout();plt.show()


## Depth-change proxy versus tilt magnitude

Each point is one eddy-day. The left panel retains the sign of the broad gradient; the right asks whether its magnitude increases with tilt. Descriptive Spearman coefficients are shown separately for AEs and CEs. Multiple days from one eddy are not independent, so no p-values or causal claim are attached.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
rows=[]
for cyc,color in [('AE','tab:red'),('CE','tab:blue')]:
    g=snapshot.loc[snapshot.Cyc.eq(cyc)]
    axes[0].scatter(g.TiltDis,1e8*g.dwdz_proxy,s=18,alpha=.35,color=color,label=cyc)
    axes[1].scatter(g.TiltDis,1e8*g.abs_dwdz_proxy,s=18,alpha=.35,color=color,label=cyc)
    for metric in ('dwdz_proxy','abs_dwdz_proxy'):
        rho=spearmanr(g.TiltDis,g[metric]).statistic if g[metric].nunique()>1 else np.nan
        rows.append(dict(Cyc=cyc,metric=metric,eddy_days=len(g),eddies=g.Eddy.nunique(),spearman_rho=rho))
axes[0].axhline(0,color='.5',lw=.8)
axes[0].set(xlabel='TiltDis (km)',ylabel='Signed depth-change proxy (10⁻⁸ s⁻¹)')
axes[1].set(xlabel='TiltDis (km)',ylabel='Absolute depth-change proxy (10⁻⁸ s⁻¹)')
for ax in axes:ax.legend()
fig.tight_layout();plt.show()
display(pd.DataFrame(rows))


The cached maximum and minimum profiles are useful for seeing the separate upward and downward cells, but they should **not** be differentiated: their locations vary by depth. The gradient above differentiates broad averages of core-mean `w`, while the core centre and ellipse also move with depth. It is a screening diagnostic rather than a pointwise `∂w/∂z` or the complete `(f+ζ)∂w/∂z` vortex-stretching term.
